In [1]:
import hashlib
import os
import platform
import uuid
import sys

STUDENT_ID = 20244080202
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV") or "base"
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]   {sys.version.split()[0]}")

[Conda 环境] pyhomework
[学号]     20244080202
[机器指纹] 0d25ba29a288e7c8
[RUN_ID]   20244080202-0d25ba29a288e7c8
[Python]   3.10.22


## 实验 1：正则表达式的基础匹配

In [2]:
import re

text_basic = "cat on mat; later: cats and dogs"
print("数字 \\d+：", re.findall(r"\d+", text_basic))
print("空白 \\s+：", re.findall(r"\s+", text_basic))
print("单词 \\bcat\\b：", re.findall(r"\bcat\b", text_basic))

text_pets = "cat dog bird CAT"
print("cat|dog：", re.findall(r"cat|dog", text_pets))
print("a|aa|aaa：", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a：", re.findall(r"aaa|aa|a", "aaaa"))

数字 \d+： []
空白 \s+： [' ', ' ', ' ', ' ', ' ', ' ']
单词 \bcat\b： ['cat']
cat|dog： ['cat', 'dog']
a|aa|aaa： ['a', 'a', 'a', 'a']
aaa|aa|a： ['aaa', 'a']


In [3]:
text_quantifier = "ct cat caat caaat"
patterns = [
    (r"ca*t", "0 个或多个 a"),
    (r"ca+t", "1 个或多个 a"),
    (r"ca?t", "0 个或 1 个 a"),
    (r"ca{2}t", "恰好 2 个 a"),
    (r"ca{1,2}t", "1 到 2 个 a"),
]
for pattern, meaning in patterns:
    print(f"{pattern:<8} {meaning:<12} -> {re.findall(pattern, text_quantifier)}")

ca*t     0 个或多个 a     -> ['ct', 'cat', 'caat', 'caaat']
ca+t     1 个或多个 a     -> ['cat', 'caat', 'caaat']
ca?t     0 个或 1 个 a   -> ['ct', 'cat']
ca{2}t   恰好 2 个 a     -> ['caat']
ca{1,2}t 1 到 2 个 a    -> ['cat', 'caat']


## 实验 2：分组、回溯引用与替换

In [4]:
# 1. 捕获组：用户名与域名
text_email = "联系：alice@example.com；备选：bob@test.org"
email_pattern = re.compile(r"(?P<user>[\w.+-]+)@(?P<domain>[\w.-]+\.[A-Za-z]{2,})")
for match in email_pattern.finditer(text_email):
    print(f"用户名 = {match.group('user'):<8} 域名 = {match.group('domain')}")

# 2. 后向引用：连续重复的单词
text_words = "cat cat, dog cat, bird bird"
print("连续重复单词：", re.findall(r"\b(\w+)\s+\1\b", text_words))

# 3. 日期格式：年-月-日 -> 月-日-年
dates = "2024-05-01\n2024/05/02\n2024.05.03"
date_pattern = r"(\d{4})([-/.])(\d{2})\2(\d{2})"
converted = re.sub(date_pattern, r"\3-\4-\1", dates)
print("转换前：")
print(dates)
print("转换后：")
print(converted)

# 4. 电话号码统一为 (123) 456-7890
phones = "联系我：123-456-7890 或 987.654.3210"
phone_pattern = r"(\d{3})[-.](\d{3})[-.](\d{4})"
print("统一后：", re.sub(phone_pattern, r"(\1) \2-\3", phones))

用户名 = alice    域名 = example.com
用户名 = bob      域名 = test.org
连续重复单词： ['cat', 'bird']
转换前：
2024-05-01
2024/05/02
2024.05.03
转换后：
05-01-2024
05-02-2024
05-03-2024
统一后： 联系我：(123) 456-7890 或 (987) 654-3210


## 实验 3：使用 BeautifulSoup 解析 HTML

In [5]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

# 1. 标题与 h1
print("页面标题：", soup.title.get_text(strip=True))
h1 = soup.find("h1")
print("h1 标签名：", h1.name)
print("h1 文本：", h1.get_text(strip=True))

# 2. class="course" 的段落与全部链接
courses = soup.find_all("p", class_="course")
links = soup.find_all("a")
print(f"\n课程段落（{len(courses)} 个）：")
for p in courses:
    print(" -", p.get_text(strip=True))
print(f"\n链接（{len(links)} 个）：")
for a in links:
    print(f" - 文字：{a.get_text(strip=True):<6} href：{a['href']}")

# 3. 不同范围的可见文本
print("\n整个文档：", soup.get_text(" ", strip=True))
print("单个段落：", courses[0].get_text(" ", strip=True))
print("单个链接：", links[0].get_text(" ", strip=True))

页面标题： 实验课程
h1 标签名： h1
h1 文本： 数据分析

课程段落（2 个）：
 - 第六讲：正则表达式
 - 第七讲：结构化数据

链接（2 个）：
 - 文字：讲义六    href：https://example.org/lecture06
 - 文字：讲义七    href：https://example.org/lecture07

整个文档： 实验课程 数据分析 第六讲：正则表达式 第七讲：结构化数据 讲义六 讲义七
单个段落： 第六讲：正则表达式
单个链接： 讲义六
